In [ ]:
import MDAnalysis as mda
from matplotlib import pyplot
import numpy as np

In [ ]:
# RMSF. Uses results from the gromacs rmsf tool

def plot_rmsf_group(files, labels, title, save=""):
    fig, ax = pyplot.subplots()
    for f, l in zip(files, labels):
        path = "/home/ONID.OREGONSTATE.EDU/waldot/1433_BAD/" + f
        data = np.loadtxt(path, comments=["#", "@"])
        ax.plot(data[:,0], data[:,1] * 10, label=l) # *10 converts to angstroms
    ax.legend(loc="upper right", fontsize=11)
    ax.tick_params(labelsize=12)
    #ax.set_title(title)
    #ax.set_xlabel("Residue #")
    #ax.set_ylabel("RMSF (nm)")
    ax.set_ylim(0, 6)
    fig.show()
    if len(save) > 0:
        fig.savefig(save)

def compute_averaged_rmsf(files):
    datasets = [np.loadtxt("/home/ONID.OREGONSTATE.EDU/waldot/1433_BAD/" + f, comments=['#', '@']) for f in files]
    final = np.zeros((datasets[0].shape[0], 3))
    stacked = np.stack(datasets, axis=0)
    final[:,0] = datasets[0][:,0]
    final[:,1] = np.mean(stacked, axis=0)[:,1]
    final[:,2] = np.std(stacked, axis=0)[:,1]
    return final

def plot_averaged_rmsf(datasets, labels, title, save=""):
    fig, ax = pyplot.subplots()
    for dataset, l in zip(datasets, labels):
        ax.plot(dataset[:,0], dataset[:,1] * 10, label=l, alpha=0.5) # *10 converts to angstroms
    ax.legend(loc="upper right", fontsize=11)
    ax.tick_params(labelsize=12)
    #ax.set_title(title)
    #ax.set_xlabel("Residue #")
    #ax.set_ylabel("RMSF (A)")
    ax.set_ylim(0, 6)
    fig.show()
    if len(save) > 0:
        fig.savefig(save)

files = ["dimer/longer_BAD_TIP3P/1433bb_rmsf_start200ns.xvg", "dimer/longer_BAD_TIP3P/rep2/1433bb_rmsf_start200ns.xvg", "dimer/longer_BAD_TIP3P/rep3/1433bb_rmsf_start200ns.xvg", "dimer/longer_BAD_TIP3P/rep4/1433bb_rmsf_start200ns.xvg"]
labels = ["Replicate 1", "Replicate 2", "Replicate 3", "Replicate 4"]
plot_rmsf_group(files, labels, "dimer", save="dimer_rmsf.png")
dimer_avg = compute_averaged_rmsf(files)

files = ["monomer/longer_BAD_TIP3P/1433bb_rmsf_start200ns.xvg", "monomer/longer_BAD_TIP3P/rep2/1433bb_rmsf_start200ns.xvg", "monomer/longer_BAD_TIP3P/rep3/1433bb_rmsf_start200ns.xvg", "monomer/longer_BAD_TIP3P/rep4/1433bb_rmsf_start200ns.xvg"]
labels = ["Replicate 1", "Replicate 2", "Replicate 3", "Replicate 4"]
plot_rmsf_group(files, labels, "monomer", save="monomer_rmsf.png")
monomer_avg = compute_averaged_rmsf(files)

plot_averaged_rmsf([dimer_avg, monomer_avg], ["Heterodimers", "Monomers"], "Averaged RMSF across replicates", save="averaged_rmsf.png")

# Stuff below generates a .defattr file that can be imported into ChimeraX to color residues based on RMSF changes

diff = monomer_avg
diff[:,1] = monomer_avg[:,1] - dimer_avg[:,1]
plot_averaged_rmsf([diff], ["diff"], "Diff")

f = open("delta_rmsf.defattr","w")
f.write("attribute: percentExposed\nmatch mode: 1-to-1\nrecipient: residues\n")
for (res, delta, _) in diff:
    f.write("\t/B:" + str(int(res)) + "\t" + str(delta) + "\n")
f.close()

In [ ]:
# Helical bends

from MDAnalysis.analysis import helix_analysis as helanal

def compute_helix_bend(sim, sel):
    # Weird way I chose to deal with file structure
    parent = sim.replace("/rep2","").replace("/rep3","").replace("/rep4","")
    prefix = "/home/ONID.OREGONSTATE.EDU/waldot/1433_BAD/"
    u = mda.Universe(prefix + parent + "/equil_50ns_pro.pdb", prefix + sim + "/production_fit.xtc")
    
    h = helanal.HELANAL(u, select=sel).run()
    return np.mean(h.results.local_bends[2000:,:], axis=0) # Ignores first 10% of trajectory (200ns)

# roi is residue numbers that were passed to helanal in selector
def plot_bends(grouped_bends, roi, title="", labels=["Replicate 1", "Replicate 2", "Replicate 3", "Replicate 4"], save=None):
    fig, ax = pyplot.subplots()
    for bends, l in zip(grouped_bends, labels):
        # Magic numbers here deal with the way helanal uses a sliding window to compute bends. See methods
        ax.plot(list(range(roi[0] + 3, roi[1] - 5 + 3)), bends, label=l)
    ax.legend(loc="upper left", fontsize=11)
    #ax.set_title(title)
    #ax.set_ylabel("Average Helical Bend (degrees)")
    #ax.set_xlabel("14-3-3Z Residue Number")
    ax.set_ylim(0,25)
    ax.tick_params(labelsize=12)
    #ax.vlines(x=49, ymin=0, ymax=25, color="yellow", linestyle="dashed")
    #ax.vlines(x=53, ymin=0, ymax=25, color="violet", linestyle="dashed")
    fig.show()

    if save is not None:
        fig.savefig(save)

sims = ["monomer/longer_BAD_TIP3P", "monomer/longer_BAD_TIP3P/rep2", "monomer/longer_BAD_TIP3P/rep3", "monomer/longer_BAD_TIP3P/rep4"]
selector = 'name CA and resnum 38-68'
monomer_bends = [compute_helix_bend(sim, selector) for sim in sims]
plot_bends(monomer_bends, (38, 68), title="14-3-3Z Helix 3 Bend, Monomer Simulations", save="monomer_bends.png")

sims = ["dimer/longer_BAD_TIP3P", "dimer/longer_BAD_TIP3P/rep2", "dimer/longer_BAD_TIP3P/rep3", "dimer/longer_BAD_TIP3P/rep4"]
selector = 'name CA and resnum 38-68 and prop id > 4000' # id > 4000 keeps 14-3-3 epsilon residues from being selected
dimer_bends = [compute_helix_bend(sim, selector) for sim in sims]
plot_bends(dimer_bends, (38, 68), title="14-3-3Z Helix 3 Bend, Dimer Simulations", save="dimer_bends.png")

monomer_avg_bends = np.mean(np.stack(monomer_bends, axis=0), axis=0)
dimer_avg_bends = np.mean(np.stack(dimer_bends, axis=0), axis=0)
plot_bends([monomer_avg_bends, dimer_avg_bends], (38, 68), title="Bend of Helix 3, Averaged Across Replicates", labels=["monomer","dimer"], save="averaged_bends.png")

In [ ]:
# Helical propensity

from MDAnalysis.analysis.dssp import DSSP

# core analysis function
# roi is atom numbers directly from sim file
def compute_ss_tendency(sim, roi):
    parent = sim.replace("/rep2","").replace("/rep3","").replace("/rep4","")
    prefix = "/home/ONID.OREGONSTATE.EDU/waldot/1433_BAD/"
    u = mda.Universe(prefix + parent + "/equil_50ns_pro.pdb", prefix + sim + "/production_fit.xtc")

    # Convert atom numbers to indices, get atoms, run
    roi = [x-1 for x in roi] 
    atoms = u.atoms[roi]
    dssp = DSSP(atoms).run()
    
    sec_structure = dssp.results.dssp
    proportion_helix = np.mean(np.where(sec_structure == 'H', 1, 0), axis=0)
    proportion_sheet = np.mean(np.where(sec_structure == 'E', 1, 0), axis=0)
    return proportion_helix, proportion_sheet

# does analysis over replicates and averages
def average_ss_over_replicates(sims, roi):
    replicates = [compute_ss_tendency(sim, roi) for sim in sims]
    rep_h = [r[0] for r in replicates]
    rep_s = [r[1] for r in replicates]
    avg_h = np.mean(np.stack(rep_h, axis=0), axis=0)
    avg_s = np.mean(np.stack(rep_s, axis=0), axis=0)
    return avg_h, avg_s

# plots helix and sheet tendency. Not used in paper
def ss_seq_plot(result, title="", res_lower=147, res_upper=164, save=None):
    fig, ax = pyplot.subplots()
    ax.plot(list(range(res_lower, res_upper+1)), result[0], c='orange', label='helix')
    ax.plot(list(range(res_lower, res_upper+1)), result[1], c='blue', label='sheet')
    ax.set_ylabel("Proportion of time in secondary structure")
    ax.set_xlabel("Residue number")
    ax.set_ylim(0,1)
    ax.legend(loc='upper right')
    ax.set_title(title)
    fig.show()
    if save is not None:
        fig.save(save)

sims = ["monomer/longer_BAD_TIP3P", "monomer/longer_BAD_TIP3P/rep2", "monomer/longer_BAD_TIP3P/rep3", "monomer/longer_BAD_TIP3P/rep4"]
monomer_res = average_ss_over_replicates(sims, list(range(4280, 4771+1)))
#ss_seq_plot(monomer_res, title="Longer BAD, monomers", res_lower=145, res_upper=175)

sims = ["dimer/longer_BAD_TIP3P", "dimer/longer_BAD_TIP3P/rep2", "dimer/longer_BAD_TIP3P/rep3", "dimer/longer_BAD_TIP3P/rep4"]
dimer_res = average_ss_over_replicates(sims, list(range(7986, 8477+1)))
#ss_seq_plot(dimer_res, title="Longer BAD, dimers", res_lower=145, res_upper=175)

# Plot used in paper
fig, ax = pyplot.subplots()
x = list(range(145,176))
ax.plot(x, dimer_res[0], label="Heterodimer Simulations")
ax.plot(x, monomer_res[0], label="Monomer Simulations")
#ax.set_xlabel("BAD Residue Number")
#ax.set_ylabel("Proportion of Time in Alpha Helix")
#ax.set_title("BAD C-terminal Helical Propensity")
ax.legend(loc="upper right", fontsize=11)
ax.tick_params(labelsize=12)
fig.show()

In [ ]:
# Interaction frequency between BAD pSer136 and BAD Arg133

# roi is atom numbers directly from file. (pSer136 phosphate, Arg133 CZ)
def compute_R133_interactions(sim, roi):
    parent = sim.replace("/rep2","").replace("/rep3","").replace("/rep4","")
    prefix = "/home/ONID.OREGONSTATE.EDU/waldot/1433_BAD/"
    u = mda.Universe(prefix + parent + "/equil_50ns_pro.pdb", prefix + sim + "/production_fit.xtc")
    
    roi = [x-1 for x in roi] # convert atom numbers to indices
    ser136 = u.atoms[roi[0]]
    arg133 = u.atoms[roi[1]]
    
    dist = np.zeros(len(u.trajectory))
    for ts in u.trajectory:
        dist[ts.frame] = np.linalg.norm(ser136.position - arg133.position)

    p_int = np.mean(np.where(dist < 5, 1, 0)) # 5A cutoff for considered interacting. Phosphate to zeta carbon    
    return p_int

def stacked_bar_plot(g1, g2, labels=["1", "2", "3", "4"], title=""):
    fig, ax = pyplot.subplots()
    
    for l, p in zip(labels, g1):
        l = "Rep. " + l
        if l == "Rep. 1":
            ax.bar(l, p, 0.75, color="tab:blue", label="Heterodimers")
        else:
            ax.bar(l, p, 0.75, color="tab:blue")

    for l, p in zip(labels, g2):
        l = "Rep. " + l + " "
        if l == "Rep. 1 ":
            ax.bar(l, p, 0.75, color="tab:orange", label="Monomers")
        else:
            ax.bar(l, p, 0.75, color="tab:orange")

    #ax.set_ylabel("Proportion of time interacting")
    #ax.set_xlabel("Simulation")
    #ax.set_title(title)
    ax.tick_params(labelsize=12)
    ax.set_ylim((0,1))
    ax.legend(loc="upper left", fontsize=11)
    
    fig.show()

sims = ["monomer/longer_BAD_TIP3P", "monomer/longer_BAD_TIP3P/rep2", "monomer/longer_BAD_TIP3P/rep3", "monomer/longer_BAD_TIP3P/rep4"]
roi = (4159, 4107) #pSer136 and Arg133
mon_res_136 = [compute_R133_interactions(sim, roi) for sim in sims]

sims = ["dimer/longer_BAD_TIP3P", "dimer/longer_BAD_TIP3P/rep2", "dimer/longer_BAD_TIP3P/rep3", "dimer/longer_BAD_TIP3P/rep4"]
roi = (7865, 7813) #pSer136 and Arg133
dim_res_136 = [compute_R133_interactions(sim, roi) for sim in sims]

stacked_bar_plot(dim_res_136, mon_res_136, title="Time Arg133 interacts with pSer136")

In [ ]:
# Interaction frequency between BAD pSer136 and 14-3-3Z Arg56

# roi is atom numbers directly from file. (BAD pSer136 phosphate, 14-3-3 Arg56 CZ)
def compute_R56_interaction(sim, roi):
    parent = sim.replace("/rep2","").replace("/rep3","").replace("/rep4","")
    prefix = "/home/ONID.OREGONSTATE.EDU/waldot/1433_BAD/"
    u = mda.Universe(prefix + parent + "/equil_50ns_pro.pdb", prefix + sim + "/production_fit.xtc")
    
    roi = [x-1 for x in roi] # convert atom numbers to indices
    ser136 = u.atoms[roi[0]]
    arg56 = u.atoms[roi[1]]
    
    dist = np.zeros((len(u.trajectory), 2))
    for ts in u.trajectory:
        dist[ts.frame, 0] = ts.time / 1000.0
        dist[ts.frame, 1] = np.linalg.norm(ser136.position - arg56.position)

    return dist

def plot_arg56_hist(mon_res, dim_res, title=""):
    mon_concat = np.concatenate(mon_res, axis=0)
    dim_concat = np.concatenate(dim_res, axis=0)
    fig, ax = pyplot.subplots()
    ax.hist(dim_concat[:,1], bins=[i/30.0 for i in range(90,240)], label="Hetereodimers")
    ax.hist(mon_concat[:,1], bins=[i/30.0 for i in range(90,240)], label="Monomers", alpha=0.6)
    ax.legend(loc="upper right", fontsize=11)
    ax.tick_params(labelsize=12)
    ax.set_title(title)
    fig.show()

sims = ["monomer/longer_BAD_TIP3P", "monomer/longer_BAD_TIP3P/rep2", "monomer/longer_BAD_TIP3P/rep3", "monomer/longer_BAD_TIP3P/rep4"]
roi = (4159, 851) #pSer136 and Arg56
mon_res_56 = [compute_R56_interaction(sim, roi) for sim in sims]

sims = ["dimer/longer_BAD_TIP3P", "dimer/longer_BAD_TIP3P/rep2", "dimer/longer_BAD_TIP3P/rep3", "dimer/longer_BAD_TIP3P/rep4"]
roi = (7865, 4560) #pSer136 and Arg56
dim_res_56 = [compute_R56_interaction(sim, roi) for sim in sims]

plot_arg56_hist(mon_res_56, dim_res_56)